In [ ]:
!rm -rf /kaggle/working/*

In [ ]:
!pip install -q transformers==4.44.2 tokenizers datasets \
             sentencepiece rouge-score accelerate py7zr evaluate

In [ ]:
import torch
import platform
import psutil

print("=== Environment Info ===")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU count       : {torch.cuda.device_count()}")
    print(f"Current device  : {torch.cuda.current_device()}")
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  - Total memory: {props.total_memory / 1e9:.2f} GB")
else:
    print("No GPU detected. Training will run on CPU.")

# CPU RAM
ram = psutil.virtual_memory()
print(f"RAM total       : {ram.total / 1e9:.2f} GB")

print(f"Device used     : {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Platform        : {platform.platform()}")

In [ ]:
import os

DATASET_SLUG = "datasets/zanzungg/dataset"
DATA_DIR     = f"/kaggle/input/{DATASET_SLUG}"

OUTPUT_DIR   = "/kaggle/working/vit5-finetuned"
LOG_DIR      = "/kaggle/working/logs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("=== Dataset Check ===")
print(f"DATA_DIR = {DATA_DIR}")

# Check DATA_DIR exists
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

print("Files in dataset directory:")
print(os.listdir(DATA_DIR))

# Check splits
for split in ["train", "val", "test"]:
    path = f"{DATA_DIR}/{split}.jsonl"
    
    if not os.path.exists(path):
        print(f"[ERROR] {split}.jsonl not found at {path}")
        continue
    
    # Count lines (dataset size)
    with open(path, "r", encoding="utf-8") as f:
        num_lines = sum(1 for _ in f)
    
    print(f"[OK] {split}.jsonl — {num_lines} samples")

print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Log directory   : {LOG_DIR}")

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                data.append(obj)
            except json.JSONDecodeError:
                print(f"[WARNING] JSON decode error at line {i} in {path}")
    return data


train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data   = load_jsonl(f"{DATA_DIR}/val.jsonl")
test_data  = load_jsonl(f"{DATA_DIR}/test.jsonl")

dataset = DatasetDict({
    "train":      Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test":       Dataset.from_list(test_data),
})

print("=== Dataset Overview ===")
print(dataset)
print("\nColumns:", dataset["train"].column_names)

# Validate schema
required_keys = ["article", "summary"]
for key in required_keys:
    if key not in dataset["train"].column_names:
        raise ValueError(f"Missing required key: {key}")

# Sample inspection
sample = dataset["train"][0]
print("\n=== Sample ===")
print(f"Article length : {len(sample['article'])} chars")
print(f"Summary length : {len(sample['summary'])} chars")
print(f"Article preview: {sample['article'][:200]}...")
print(f"Summary preview: {sample['summary'][:100]}...")

# Basic statistics
def avg_len(data, key):
    return sum(len(x[key]) for x in data) / len(data)

print("\n=== Length Statistics ===")
print(f"Avg article length: {avg_len(train_data, 'article'):.1f}")
print(f"Avg summary length: {avg_len(train_data, 'summary'):.1f}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME  = "VietAI/vit5-base-vietnews-summarization"
ARTICLE_KEY = "article"
SUMMARY_KEY = "summary"

MAX_INPUT   = 1024
MAX_TARGET  = 128

PREFIX = "summarize: "

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, clean_up_tokenization_spaces=False)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.config.use_cache = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model stats
total_params     = model.num_parameters()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== Model Info ===")
print(f"Model name       : {MODEL_NAME}")
print(f"Total params     : {total_params/1e6:.1f}M")
print(f"Trainable params : {trainable_params/1e6:.1f}M")
print(f"Vocab size       : {tokenizer.vocab_size}")
print(f"Device           : {device}")

# Config insight
print("\n=== Model Config ===")
print(model.config)

In [ ]:
def preprocess(examples):
    inputs  = ["summarize: " + doc for doc in examples[ARTICLE_KEY]]
    targets = examples[SUMMARY_KEY]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = dataset.map(
    preprocess,
    batched=True,
    batch_size=512,
    num_proc=2,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)

print("\nTokenization completed!")
print(tokenized)

# Debug sample
sample = tokenized["train"][0]

print("\n=== Tokenized Sample ===")
print(f"Input length : {len(sample['input_ids'])}")
print(f"Label length : {len(sample['labels'])}")

# Decode check
decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
print(f"\nDecoded input preview:\n{decoded_input[:200]}...")

In [ ]:
import evaluate
import numpy as np

# 1. Định nghĩa hàm tính ROUGE
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    preds = preds.astype(np.int32) 
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = labels.astype(np.int32)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True, clean_up_tokenization_spaces=True)

    # Tính ROUGE
    result = rouge_metric.compute(
        predictions=decoded_preds, 
        references=decoded_labels, 
        use_stemmer=True
    )
    
    # Làm đẹp kết quả (nhân 100)
    result = {k: round(v * 100, 4) for k, v in result.items()}
    return result

In [ ]:
import math
import torch
from transformers import Seq2SeqTrainingArguments

num_gpus = max(1, torch.cuda.device_count())

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Chiến lược Training
    num_train_epochs=3,
    per_device_train_batch_size=8,      # Tăng lên 8 để tận dụng 16GB VRAM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # Effective batch size = 8*2*2 = 32

    save_safetensors=False,

    # Tối ưu hóa bộ nhớ
    fp16=True,                          # Bắt buộc cho Tesla T4
    gradient_checkpointing=True,        # Tiết kiệm VRAM, cho phép tăng batch size
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Tối ưu hóa hội tụ
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # Đánh giá & Lưu trữ
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    
    # Generation (trong lúc eval)
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    generation_num_beams=4,             # Tăng lên 4 để tóm tắt chất lượng hơn
    
    # Khác
    logging_steps=50,
    report_to="none",                   # Có thể dùng "wandb" nếu muốn theo dõi biểu đồ
    group_by_length=True,               # Tăng tốc độ train đáng kể
    dataloader_num_workers=4,
)

n_train = len(tokenized["train"])
batch_eff = (
    training_args.per_device_train_batch_size
    * num_gpus
    * training_args.gradient_accumulation_steps
)

steps_per_epoch = math.ceil(n_train / batch_eff)
total_steps     = steps_per_epoch * training_args.num_train_epochs

print("=== Training Setup ===")
print(f"Train samples     : {n_train:,}")
print(f"Effective batch   : {batch_eff}")
print(f"Steps / epoch     : {steps_per_epoch:,}")
print(f"Total steps       : {total_steps:,}")
print(f"Warmup ratio      : {training_args.warmup_ratio}")

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import torch

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if training_args.fp16 else None,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Logging
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("=== Training Start ===")
print(f"Model        : {MODEL_NAME}")
print(f"Device       : {gpu_name}")
print(f"Epochs       : {training_args.num_train_epochs}")
print(f"FP16         : {training_args.fp16}")
print(f"Beam size    : {training_args.generation_num_beams}")

print("\nStarting full training...")
trainer.train()

In [ ]:
import numpy as np

print("Running evaluation on test set...")
test_results = trainer.predict(
    tokenized["test"],
    metric_key_prefix="test",
)
metrics = test_results.metrics

print("\n" + "="*50)
print("         TEST SET RESULTS")
print("="*50)
for k in ["test_rouge1", "test_rouge2", "test_rougeL"]:
    print(f"{k:15}: {metrics.get(k, 0):.2f}")
print("="*50)

preds = test_results.predictions
labels = test_results.label_ids

preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

print("\n=== Sample Predictions ===")
for i in range(3):
    print(f"\n[Sample {i+1}]")
    original_article = dataset["test"][i]["article"]
    
    print(f"[-] Article (shortened): {original_article[:200]}...")
    print(f"[-] Ground Truth       : {decoded_labels[i]}")
    print(f"[-] Model Prediction   : {decoded_preds[i]}")
    print("-" * 30)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import json
with open(f"{OUTPUT_DIR}/test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nModel and metrics saved to: {OUTPUT_DIR}")